In [33]:
import pickle
import numpy as np
from FastBEMT.JobParameters import AerodynamicParameters, AcousticParameters
from FastBEMT.Propeller import Propeller
import aerosandbox as asb

with open("../Datasets/Propellers/10x5E.pkl", "rb") as f:
    blade_dict = pickle.load(f)

aerodynamic_params = AerodynamicParameters(
    prop_radius=blade_dict['tip_radius'],
    hub_radius=blade_dict['hub_radius'],
    n_blades=blade_dict['n_blades'],
    rpm=7000,
    a_inf=343,
    rho=1.225,
    mu=1.81e-5,
)

acoustic_params = AcousticParameters(
    aero_params=aerodynamic_params,
    p_ref=2e-5, revolutions=100, num_obs_times_per_rev=100
)

In [34]:
propeller = Propeller(
    propeller_geometry=blade_dict,
    aero_params=aerodynamic_params,
    acoustic_params=acoustic_params
)
v_inf = 0
propeller.run_bemt(v_inf=v_inf)

In [38]:
propeller.compute_total_forces()

(np.float64(6.973623881714827),
 np.float64(0.100741361660329),
 np.float64(0.1004833223043184),
 np.float64(0.03590785261635691))

In [35]:
import timeit

n_iterations = 20

def measure():
    propeller.run_bemt(v_inf=v_inf)

avg_time = timeit.timeit(measure, number=n_iterations) / n_iterations

print(f"Average BEMT execution time over {n_iterations} runs: {avg_time:.4f} seconds")

Average BEMT execution time over 20 runs: 0.4289 seconds


In [36]:
def prandtlLoss(phi_grid, r_vector):
    B = aerodynamic_params.n_blades
    R_tip = aerodynamic_params.prop_radius
    R_hub = aerodynamic_params.hub_radius
    
    r = r_vector[:, np.newaxis] 
    phi = phi_grid[np.newaxis, :]
    denom = 2 * r * np.sin(phi)
    f_tip = B * (R_tip - r) / denom
    f_hub = B * (r - R_hub) / denom
    F_tip = (2/np.pi) * np.arccos(np.exp(-np.clip(f_tip, 0, 50)))
    F_hub = (2/np.pi) * np.arccos(np.exp(-np.clip(f_hub, 0, 50)))
    
    return F_tip * F_hub

In [ ]:
from scipy.interpolate import RegularGridInterpolator

n_stations = len(blade_dict['r'])
n_blades = aerodynamic_params.n_blades

_tables = [{} for _ in range(n_stations)]

modelSize = 'xxxlarge'
initial_phi = propeller.solution_data.phi.values

T_ts = 0
Q_ts = 0
r = np.array(blade_dict['r'])
vT = aerodynamic_params.omega * r   
vQs = np.ones_like(vT)
dr = np.array(blade_dict['dr'])
chord = np.array(blade_dict['chord'])
twist = np.array(blade_dict['twist'])
alpha_grid = np.linspace(-30.0, 40.0, 141)
phi_table = np.linspace(np.radians(0.1), np.radians(89.9), 100)
F_table = prandtlLoss(phi_table, r)
interp_func = RegularGridInterpolator((r, phi_table), F_table, bounds_error=False, fill_value=None)
# --- 2. SIMULATION LOOP ---
for time_idx, t in enumerate(range(100)):
    print(time_idx)
    phi = np.arctan2(vQs, vT)
    alpha = twist - np.degrees(phi)
    W = np.sqrt(vQs**2 + vT**2)
    
    Re = aerodynamic_params.rho * W * chord / aerodynamic_params.mu
    Ma = W / aerodynamic_params.a_inf
    
    ReBin = np.round(Re, -4)
    MaBin = np.round(Ma, 1)
    keys = list(zip(ReBin, MaBin))

    for i, airfoil in enumerate(blade_dict['airfoil']):
        key = keys[i]
        if key not in _tables[i]:        
            full_output = asb.Airfoil.get_aero_from_neuralfoil(asb.Airfoil(coordinates=airfoil), alpha=alpha_grid, Re=ReBin[i], mach=MaBin[i], model_size=modelSize)
            cl_grid = np.asarray(full_output["CL"], dtype=float)
            cd_grid = np.asarray(full_output["CD"], dtype=float)
            _tables[i][key] = (alpha_grid, cl_grid, cd_grid)

    alpha_grid, cl_grid, cd_grid = _tables[i][key]
    cL = np.interp(alpha, alpha_grid, cl_grid)
    cD = np.interp(alpha, alpha_grid, cd_grid)

    nan_mask = np.isnan(cL) | np.isnan(cD)

    if np.any(nan_mask):
        bad_alphas = alpha[nan_mask]
        bad_Res = ReBin[nan_mask]
        bad_Mas = MaBin[nan_mask]
        for i in np.where(nan_mask)[0]:
            full_output = asb.Airfoil.get_aero_from_neuralfoil(
                blade_dict['airfoil'][i],
                alpha=bad_alphas[i], 
                Re=bad_Res[i], 
                mach=bad_Mas[i], 
                model_size=modelSize
            )
            
            # 3. Fill the NaNs back into the original arrays
            cL[nan_mask] = full_output["CL"]
            cD[nan_mask] = full_output["CD"]

    cLPrime = cL * np.cos(phi) - cD * np.sin(phi)
    cDPrime = cL * np.sin(phi) + cD * np.cos(phi)
        
    F = interp_func(np.column_stack((r, phi)))
    print(F)
    # F = 1
    sigma = aerodynamic_params.n_blades * chord / (2 * np.pi * r)
    k_t = sigma * cLPrime / (4 * F * np.sin(phi) * np.cos(phi)) 
    k_q = sigma * cDPrime / (4 * F * np.sin(phi) * np.cos(phi))
    aPrime = k_q / (1 + k_q)
    vQs_new = aerodynamic_params.omega * r * k_t / (1 + k_q) + v_inf
    vT_new = aerodynamic_params.omega * r * (1 - aPrime)

    vQs += 0.7 * (vQs_new - vQs)
    vT += 0.7 * (vT_new - vT)

dT = 0.5 * aerodynamic_params.rho * W**2 * chord * cLPrime * dr * n_blades
dQ = 0.5 * aerodynamic_params.rho * W**2 * chord * cDPrime * r * dr * n_blades

print(np.sum(dT))


0


ValueError: The requested sample points xi have dimension 38 but this RegularGridInterpolator has dimension 2